# 统一训练流程 - 索引 + 模型 + 上传

**功能**:
1. 构建 FAISS 索引
2. Fine-tune 模型（可选）
3. 自动上传到 HuggingFace Hub
4. 评估对比 Base vs Fine-tuned

**路径**: 使用新的 `Epiq Project/pipeline` 结构

**运行环境**: Google Colab with GPU

## Step 0: Setup

In [ ]:
# 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 克隆代码
import os
os.chdir('/content')

if not os.path.exists('RAG-embedding'):
    !git clone https://github.com/Echo-Lee/RAG-embedding.git
else:
    !cd RAG-embedding && git pull

os.chdir('RAG-embedding')
print(f"✅ Current directory: {os.getcwd()}")

In [ ]:
# 安装依赖
!pip install -q sentence-transformers faiss-cpu pyyaml tqdm huggingface_hub datasets

import sys
sys.path.insert(0, 'src')

print("✅ Dependencies installed")

## Step 1: 配置

In [ ]:
from pathlib import Path

# ========== 路径配置 ==========
DRIVE_BASE = Path("/content/drive/MyDrive/Epiq Project/pipeline")

# 数据路径
DATA_PATHS = {
    'hospital': DRIVE_BASE / "data/processed/hospital/threads_with_summary.json",
    'corruption': DRIVE_BASE / "data/processed/corruption/emails_group_by_thread.json"
}

# 输出路径
FAISS_OUTPUT_DIR = DRIVE_BASE / "faiss_index"
MODEL_OUTPUT_DIR = DRIVE_BASE / "models"

FAISS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ========== 模型配置 ==========
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
BATCH_SIZE = 32

# ========== HF Hub 配置 ==========
YOUR_HF_USERNAME = "YOUR_USERNAME"  # ⚠️ 修改这里
HF_INDEX_REPO = f"{YOUR_HF_USERNAME}/rag-indexes"

# ========== 任务选择 ==========
DATASETS = ['hospital', 'corruption']  # 要处理的数据集
BUILD_INDEX = True  # 是否构建索引
TRAIN_MODEL = False  # 是否训练模型（需要 question-summary pairs）
UPLOAD_TO_HF = True  # 是否上传到 HF Hub

print("📁 Configuration:")
print(f"  Drive base: {DRIVE_BASE}")
print(f"  FAISS output: {FAISS_OUTPUT_DIR}")
print(f"  Model output: {MODEL_OUTPUT_DIR}")
print(f"  HF username: {YOUR_HF_USERNAME}")
print(f"  Tasks: Build={BUILD_INDEX}, Train={TRAIN_MODEL}, Upload={UPLOAD_TO_HF}")

## Step 2: 构建 FAISS 索引

In [ ]:
if BUILD_INDEX:
    from data.loader import DataLoader
    from config.config import RAGConfig, DatasetConfig
    import faiss
    from sentence_transformers import SentenceTransformer
    import json
    from datetime import datetime
    from tqdm import tqdm

    def build_index_for_dataset(dataset_name):
        print(f"\n{'='*60}")
        print(f"🔨 Building index for {dataset_name}")
        print(f"{'='*60}")

        # 加载数据
        dataset_config = DatasetConfig(
            name=dataset_name,
            data_path=DATA_PATHS[dataset_name]
        )
        config = RAGConfig(
            dataset=dataset_config,
            embedding_model=EMBEDDING_MODEL,
            batch_size=BATCH_SIZE
        )

        loader = DataLoader(config)
        documents = loader.load()
        print(f"✅ Loaded {len(documents)} documents")

        # 加载模型
        model = SentenceTransformer(EMBEDDING_MODEL)
        model.max_seq_length = 512

        # 编码文档
        print("\n⏳ Encoding documents...")
        texts = [doc['content'] for doc in documents]
        embeddings = model.encode(texts, batch_size=BATCH_SIZE, 
                                  show_progress_bar=True, normalize_embeddings=True)

        # 构建索引
        print("\n🔨 Building FAISS index...")
        dimension = embeddings.shape[1]
        index = faiss.IndexFlatIP(dimension)
        index.add(embeddings)

        print(f"✅ Index built: {index.ntotal} vectors, dim={dimension}")

        # 保存
        output_dir = FAISS_OUTPUT_DIR / dataset_name
        output_dir.mkdir(parents=True, exist_ok=True)

        index_path = output_dir / "faiss_index.bin"
        faiss.write_index(index, str(index_path))
        print(f"💾 Saved: {index_path}")

        # 保存元数据
        metadata = {
            'dataset_name': dataset_name,
            'num_docs': len(documents),
            'dimension': dimension,
            'index_type': 'IndexFlatIP',
            'embedding_model': EMBEDDING_MODEL,
            'created_at': datetime.now().isoformat()
        }
        with open(output_dir / "metadata.json", 'w') as f:
            json.dump(metadata, f, indent=2)
        print(f"💾 Saved: {output_dir / 'metadata.json'}")

        return output_dir, documents

    # 构建所有数据集
    built_indexes = {}
    for dataset in DATASETS:
        try:
            output_dir, docs = build_index_for_dataset(dataset)
            built_indexes[dataset] = {'path': output_dir, 'docs': docs}
        except Exception as e:
            print(f"❌ Failed: {e}")
            import traceback
            traceback.print_exc()

    print(f"\n✅ Built {len(built_indexes)} indexes")
else:
    print("⏭️  Skipping index building")

## Step 3: Fine-tune 模型（可选）

In [ ]:
if TRAIN_MODEL:
    print("🚀 Starting model fine-tuning...")

    # 检查是否有 question-summary pairs
    QA_PAIRS_PATH = DRIVE_BASE / "data/processed/thread_questions.json"  # 你需要提供这个文件

    if not QA_PAIRS_PATH.exists():
        print(f"❌ QA pairs not found at {QA_PAIRS_PATH}")
        print("   Please generate question-summary pairs first")
        TRAIN_MODEL = False
    else:
        import random
        import torch
        from datasets import Dataset as HFDataset
        from sentence_transformers import SentenceTransformer
        from sentence_transformers.losses import TripletLoss
        from sentence_transformers.training_args import SentenceTransformerTrainingArguments
        from sentence_transformers import SentenceTransformerTrainer

        # 加载 QA pairs
        with open(QA_PAIRS_PATH) as f:
            thread_qa = json.load(f)

        pairs = []
        for tid, rec in thread_qa.items():
            q = rec.get("question", "").strip()
            s = rec.get("summary", "").strip()
            if q and s:
                pairs.append({"thread_id": tid, "question": q, "summary": s})

        print(f"✅ Loaded {len(pairs)} QA pairs")

        # 划分训练/测试集
        random.seed(42)
        random.shuffle(pairs)
        n_test = min(150, len(pairs) // 5)
        test_pairs = pairs[:n_test]
        train_pairs = pairs[n_test:]
        print(f"Train: {len(train_pairs)}, Test: {len(test_pairs)}")

        # 构建 triplets
        summaries_by_tid = {p["thread_id"]: p["summary"] for p in pairs}
        tids = list(summaries_by_tid.keys())

        train_triplets = []
        for p in train_pairs:
            tid = p["thread_id"]
            neg_tid = random.choice([t for t in tids if t != tid])
            train_triplets.append({
                "anchor": p["question"],
                "positive": p["summary"],
                "negative": summaries_by_tid[neg_tid]
            })

        train_dataset = HFDataset.from_dict({
            "sentence1": [t["anchor"] for t in train_triplets],
            "sentence2": [t["positive"] for t in train_triplets],
            "sentence3": [t["negative"] for t in train_triplets]
        })

        # 加载模型并训练
        model = SentenceTransformer(EMBEDDING_MODEL, device="cuda" if torch.cuda.is_available() else "cpu")
        model.max_seq_length = 512

        train_loss = TripletLoss(model=model)

        finetuned_dir = MODEL_OUTPUT_DIR / "finetuned-hospital-v1"

        args = SentenceTransformerTrainingArguments(
            output_dir=str(finetuned_dir),
            num_train_epochs=2,
            per_device_train_batch_size=8,
            warmup_steps=50,
            fp16=torch.cuda.is_available(),
            logging_steps=20,
            save_strategy="epoch"
        )

        trainer = SentenceTransformerTrainer(
            model=model,
            args=args,
            train_dataset=train_dataset,
            loss=train_loss
        )

        print("\n⏳ Training...")
        trainer.train()
        model.save(str(finetuned_dir))
        print(f"✅ Model saved to {finetuned_dir}")

        # 评估
        print("\n📊 Evaluating...")
        import numpy as np

        def hit_at_k(model, test_pairs, corpus_tids, corpus_texts, k_values=(1, 3, 5)):
            corpus_embs = model.encode(corpus_texts, convert_to_numpy=True, normalize_embeddings=True)
            hits = {k: 0 for k in k_values}
            for p in test_pairs:
                q_emb = model.encode(p["question"], convert_to_numpy=True, normalize_embeddings=True)
                scores = np.dot(corpus_embs, q_emb.T).flatten()
                top_indices = np.argsort(scores)[::-1][:max(k_values)]
                top_tids = [corpus_tids[i] for i in top_indices]
                gold = p["thread_id"]
                for k in k_values:
                    if gold in top_tids[:k]:
                        hits[k] += 1
            return {k: 100.0 * hits[k] / len(test_pairs) for k in k_values}

        corpus_tids = [p["thread_id"] for p in pairs]
        corpus_texts = [p["summary"] for p in pairs]

        base_model = SentenceTransformer(EMBEDDING_MODEL)
        base_metrics = hit_at_k(base_model, test_pairs, corpus_tids, corpus_texts)

        ft_model = SentenceTransformer(str(finetuned_dir))
        ft_metrics = hit_at_k(ft_model, test_pairs, corpus_tids, corpus_texts)

        print("\n=== Base vs Fine-tuned ===")
        print(f"Hit@1: {base_metrics[1]:.2f}% → {ft_metrics[1]:.2f}%")
        print(f"Hit@3: {base_metrics[3]:.2f}% → {ft_metrics[3]:.2f}%")
        print(f"Hit@5: {base_metrics[5]:.2f}% → {ft_metrics[5]:.2f}%")

else:
    print("⏭️  Skipping model training")

## Step 4: 上传到 HuggingFace Hub

In [ ]:
if UPLOAD_TO_HF:
    from huggingface_hub import login, HfApi

    if YOUR_HF_USERNAME == "YOUR_USERNAME":
        print("❌ Please set YOUR_HF_USERNAME first!")
    else:
        print("\n🔐 Login to HuggingFace Hub")
        hf_token = input("Enter HF Token: ")
        login(token=hf_token)

        api = HfApi()

        # 上传索引
        if BUILD_INDEX and built_indexes:
            print(f"\n📦 Creating repository: {HF_INDEX_REPO}")
            api.create_repo(repo_id=HF_INDEX_REPO, repo_type="dataset", exist_ok=True)

            for dataset in built_indexes:
                print(f"\n📤 Uploading {dataset} index...")
                api.upload_folder(
                    folder_path=str(built_indexes[dataset]['path']),
                    repo_id=HF_INDEX_REPO,
                    repo_type="dataset",
                    path_in_repo=dataset
                )
                print(f"✅ {dataset} uploaded")

            print(f"\n✅ All indexes uploaded!")
            print(f"View at: https://huggingface.co/datasets/{HF_INDEX_REPO}")

        # 上传模型
        if TRAIN_MODEL and (MODEL_OUTPUT_DIR / "finetuned-hospital-v1").exists():
            model_repo = f"{YOUR_HF_USERNAME}/rag-finetuned-hospital"
            print(f"\n📦 Creating model repository: {model_repo}")
            api.create_repo(repo_id=model_repo, repo_type="model", exist_ok=True)

            print("\n📤 Uploading fine-tuned model...")
            api.upload_folder(
                folder_path=str(MODEL_OUTPUT_DIR / "finetuned-hospital-v1"),
                repo_id=model_repo,
                repo_type="model"
            )
            print(f"✅ Model uploaded!")
            print(f"View at: https://huggingface.co/{model_repo}")

else:
    print("⏭️  Skipping HF Hub upload")

## 完成！

### 摘要

✅ **索引**: 保存在 Google Drive `Epiq Project/pipeline/faiss_index/`

✅ **模型**: 保存在 Google Drive `Epiq Project/pipeline/models/`

✅ **HF Hub**: 索引和模型已上传（如果启用）

### 下一步

1. 访问 HuggingFace Hub 验证上传
2. 创建 Gradio Space 使用 `gradio_app_compare.py`
3. 在 Space 中添加 fine-tuned 模型配置
4. 测试双栏对比效果